# C1 · 4. Children: labelled test clips, agreement, and the %SS severity check

SEP-28k speakers are adults, so the model has to be checked on **children** separately.

**A. Child test clips (UCLASS or the team's Sinhala recordings).** Two people label the same clips independently with `tools/label_clips.py`. This part reports their agreement (**Cohen's κ** per label), keeps the clips they agree on, and scores the exported head and the rule-based detector on them.

**B. %SS severity check.** C1's %SS is compared with a reference %SS for the same recordings:
- FluencyBank Voices-CWS CHAT transcripts (`training/chat_ss.py` counts stuttered syllables the FluCalc way), when the supervisor's membership is granted; or
- hand counts from an SLP (a CSV).

Settings: `SWARA_C1_LABEL_SET` (folder with `manifest.csv`, `clips/`, `labels_A.csv`, `labels_B.csv`), `SWARA_C1_RATERS` (default `A,B`), `SWARA_C1_HEAD` (default `models/c1_head.joblib`), `SWARA_C1_CHAT_DIR` (FluencyBank `.cha` files with a 16 kHz WAV of the same name), `SWARA_C1_SS_CSV` (columns `audio`, `referencePercentSS`, optional `transcript`, `taskType`, `passageText`), `SWARA_C1_SPEAKER` (CHAT speaker code, default `CHI`).

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # the speech service's own code (app/, training/)
DATA = Path(os.environ.get("SWARA_C1_DATA", ROOT / "data" / "sep28k"))

LABEL_SET = os.environ.get("SWARA_C1_LABEL_SET")
RATERS = os.environ.get("SWARA_C1_RATERS", "A,B").split(",")
HEAD = Path(os.environ.get("SWARA_C1_HEAD", ROOT / "models" / "c1_head.joblib"))
CHAT_DIR = os.environ.get("SWARA_C1_CHAT_DIR")
SS_CSV = os.environ.get("SWARA_C1_SS_CSV")
SPEAKER = os.environ.get("SWARA_C1_SPEAKER", "CHI")
os.environ.setdefault("C1_MODEL_DIR", str(HEAD.parent))
from training import agreement, heads
LABELS = agreement.LABELS

## A1. Agreement between the two raters

In [ ]:
if LABEL_SET:
    LS = Path(LABEL_SET)
    a = agreement.read_labels(LS / f"labels_{RATERS[0]}.csv")
    b = agreement.read_labels(LS / f"labels_{RATERS[1]}.csv")
    kappa_table = agreement.kappa(a, b)
    agreed, to_resolve = agreement.consensus(a, b)
    print(f"{len(agreed)} clips agreed on every label, {len(to_resolve)} to resolve (third rater or discussion)")
    display(kappa_table.round(3))
else:
    print("SWARA_C1_LABEL_SET is not set: skip part A.")

## A2. The model on the children's clips

In [ ]:
if LABEL_SET and len(agreed):
    import joblib
    from training import features
    head = joblib.load(HEAD)
    paths = [str(LS / "clips" / c) for c in agreed.index]
    parts = []
    for source in head["inputs"]:
        if source == "acoustic":
            parts.append(features.acoustic(paths, LS / "features"))
        else:
            parts.append(features.xlsr(paths, LS / "features", head["encoder"], [head["layer"]])[head["layer"]])
    p = heads.proba(head["model"], np.concatenate(parts, axis=1))
    y_child = agreed[list(head["labels"])].to_numpy()
    child = heads.evaluate(p, y_child, head["labels"], head["thresholds"])
    rules_child = heads.evaluate(heads.rule_baseline(paths, head["labels"]), y_child, head["labels"])
    sep_test = head["training"]["metrics"]["test"]
    child_table = pd.DataFrame({
        "positives (children)": {l: child["perLabel"][l]["positives"] for l in head["labels"]},
        "F1 children": {l: child["perLabel"][l]["f1"] for l in head["labels"]},
        "F1 SEP-28k test (adults)": {l: sep_test["perLabel"][l]["f1"] for l in head["labels"]},
        "rule-based F1 children": {l: rules_child["perLabel"][l]["f1"] for l in head["labels"]},
    })
    child_table.loc["macro-F1"] = [None, child["macroF1"], sep_test["macroF1"], rules_child["macroF1"]]
    display(child_table.round(3))

## B. %SS: C1 against a reference

Each recording is run through the **service's own pipeline** (`app.pipeline.analyze`, with the detector set by `C1_DETECTOR`), and its %SS is compared with the reference: Pearson and Spearman correlation, mean difference with 95 % limits of agreement (Bland–Altman), and ICC(2,1) for absolute agreement.

In [ ]:
import io
import soundfile as sf
from app import pipeline
from training import chat_ss

def run_c1(audio: np.ndarray, sr: int, task="picture_description", transcript=None, passage=None, language="en"):
    audio = audio[: 180 * sr]  # the service's 3-minute limit
    buf = io.BytesIO(); sf.write(buf, audio, sr, format="WAV")
    return pipeline.analyze(buf.getvalue(), task, language, passage_text=passage, transcript=transcript)

pairs = []
if CHAT_DIR:
    for cha in sorted(Path(CHAT_DIR).glob("*.cha")):
        wav = cha.with_suffix(".wav")
        if not wav.exists():
            continue
        ref = chat_ss.from_file(cha, SPEAKER)
        audio, sr = sf.read(wav, dtype="float32")
        if audio.ndim > 1: audio = audio.mean(axis=1)
        # Only the child's own utterances (CHAT time bullets), joined; the interviewer is left out.
        child_audio = np.concatenate([audio[int(s / 1000 * sr): int(e / 1000 * sr)] for s, e in ref.segments_ms]) if ref.segments_ms else audio
        res = run_c1(child_audio, sr, transcript=" ".join(ref.intended))
        pairs.append({"recording": cha.stem, "referencePercentSS": ref.as_dict()["percentSS"], "c1PercentSS": res.percentSS})
if SS_CSV:
    for row in pd.read_csv(SS_CSV).to_dict("records"):
        audio, sr = sf.read(row["audio"], dtype="float32")
        if audio.ndim > 1: audio = audio.mean(axis=1)
        res = run_c1(audio, sr, row.get("taskType", "picture_description"), row.get("transcript"), row.get("passageText"),
                     row.get("language", "si"))
        pairs.append({"recording": Path(row["audio"]).stem, "referencePercentSS": row["referencePercentSS"], "c1PercentSS": res.percentSS})
ss = pd.DataFrame(pairs)
ss

In [ ]:
from scipy.stats import pearsonr, spearmanr

def icc_2_1(x, y):
    # Two-way random effects, absolute agreement, single measure (Shrout & Fleiss ICC(2,1)).
    data = np.stack([x, y], axis=1); n, k = data.shape
    grand = data.mean()
    msr = k * ((data.mean(axis=1) - grand) ** 2).sum() / (n - 1)
    msc = n * ((data.mean(axis=0) - grand) ** 2).sum() / (k - 1)
    sse = ((data - data.mean(axis=1, keepdims=True) - data.mean(axis=0, keepdims=True) + grand) ** 2).sum()
    mse = sse / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)

if len(ss) >= 3:
    x, yv = ss["referencePercentSS"].to_numpy(float), ss["c1PercentSS"].to_numpy(float)
    d = yv - x
    summary = {"recordings": len(ss), "pearson r": pearsonr(x, yv)[0], "spearman rho": spearmanr(x, yv)[0],
               "mean difference (C1 − reference)": d.mean(), "95% limits of agreement": (d.mean() - 1.96 * d.std(ddof=1), d.mean() + 1.96 * d.std(ddof=1)),
               "ICC(2,1)": icc_2_1(x, yv)}
    display(pd.Series(summary))
else:
    print("Fewer than 3 recordings with a reference %SS: nothing to compare yet.")

**Reading it.** High correlation with a constant offset means C1 ranks children correctly but its scale needs calibrating; the mean difference shows by how much. Large limits of agreement mean individual scores can't be trusted yet. The severity bands in `app/severity/config.py` are provisional until this check is done on children.